# 07. 매칭 성능 평가 (TF-IDF vs Sentence-BERT)

**목표**: Ground Truth 기반 정량 지표로 두 매칭 방법의 성능 차이를 검증

| 평가 지표 | 설명 |
|----------|------|
| **Top-1 정확도** | 1순위 추천이 정답인 비율 |
| **Top-3 정확도** | 3순위 안에 정답이 포함된 비율 |
| **MRR** | 정답 순위 역수의 평균 (1위=1.0, 2위=0.5, 3위=0.33) |

> **데이터**: `data/processed/matching_results.csv` (06_matching.ipynb 결과)

## 0. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import ast
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

from utils.config import DATA_PROCESSED, PROJECT_ROOT, is_s3, S3_BUCKET, AWS_REGION

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

RESULTS_PATH    = DATA_PROCESSED / 'matching_results.csv'
EVAL_OUTPUT     = DATA_PROCESSED / 'eval_figures'
EVAL_OUTPUT.mkdir(exist_ok=True)

print("환경 설정 완료")

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(RESULTS_PATH)

# 문자열로 저장된 리스트 파싱
df['tfidf_top5'] = df['tfidf_top5'].apply(ast.literal_eval)
df['sbert_top5'] = df['sbert_top5'].apply(ast.literal_eval)

print(f"평가 쿼리: {len(df)}개")
print(f"컬럼: {list(df.columns)}")
df[['query_id', 'lifeCycle', 'correct_idx', 'tfidf_top5', 'sbert_top5']].head(3)

## 2. 평가 지표 계산 함수

In [ ]:
def top_k_accuracy(pred_list: list[list[int]], true_list: list[int], k: int) -> float:
    """상위 k개 안에 정답이 포함된 비율."""
    correct = sum(
        1 for pred, true in zip(pred_list, true_list)
        if true in pred[:k]
    )
    return correct / len(true_list)


def mean_reciprocal_rank(pred_list: list[list[int]], true_list: list[int]) -> float:
    """정답 순위 역수의 평균 — 높을수록 상위에 정답 등장."""
    rr_list = []
    for pred, true in zip(pred_list, true_list):
        if true in pred:
            rank = pred.index(true) + 1  # 1-indexed
            rr_list.append(1.0 / rank)
        else:
            rr_list.append(0.0)
    return sum(rr_list) / len(rr_list)


# 전체 지표 계산
true_indices = df['correct_idx'].tolist()

metrics = {
    'TF-IDF': {
        'Top-1': top_k_accuracy(df['tfidf_top5'].tolist(), true_indices, k=1),
        'Top-3': top_k_accuracy(df['tfidf_top5'].tolist(), true_indices, k=3),
        'MRR':   mean_reciprocal_rank(df['tfidf_top5'].tolist(), true_indices),
    },
    'Sentence-BERT': {
        'Top-1': top_k_accuracy(df['sbert_top5'].tolist(), true_indices, k=1),
        'Top-3': top_k_accuracy(df['sbert_top5'].tolist(), true_indices, k=3),
        'MRR':   mean_reciprocal_rank(df['sbert_top5'].tolist(), true_indices),
    },
}

df_metrics = pd.DataFrame(metrics).T
print("=== 전체 성능 비교 ===")
print(df_metrics.round(4).to_string())
print()
print(f"BERT vs TF-IDF Top-1 향상: {(metrics['Sentence-BERT']['Top-1'] - metrics['TF-IDF']['Top-1'])*100:+.1f}%p")
print(f"BERT vs TF-IDF MRR 향상:   {(metrics['Sentence-BERT']['MRR']   - metrics['TF-IDF']['MRR'])*100:+.1f}%p")

## 3. 전체 성능 비교 막대 그래프

In [ ]:
labels  = ['Top-1 정확도', 'Top-3 정확도', 'MRR']
tfidf_scores = [metrics['TF-IDF']['Top-1'], metrics['TF-IDF']['Top-3'], metrics['TF-IDF']['MRR']]
sbert_scores = [metrics['Sentence-BERT']['Top-1'], metrics['Sentence-BERT']['Top-3'], metrics['Sentence-BERT']['MRR']]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, tfidf_scores, width, label='TF-IDF',        color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + width/2, sbert_scores, width, label='Sentence-BERT', color='#DD8452', alpha=0.85)

# 수치 레이블
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#DD8452')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('점수', fontsize=12)
ax.set_title('TF-IDF vs Sentence-BERT 매칭 성능 비교', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(EVAL_OUTPUT / 'performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 생애주기별 성능 분해 분석

In [ ]:
lifecycle_order = ['자견', '성견', '노령견']
lc_results = {}

for lc in lifecycle_order:
    sub = df[df['lifeCycle'] == lc]
    true_sub = sub['correct_idx'].tolist()
    lc_results[lc] = {
        'TF-IDF Top-1':       top_k_accuracy(sub['tfidf_top5'].tolist(), true_sub, k=1),
        'TF-IDF MRR':         mean_reciprocal_rank(sub['tfidf_top5'].tolist(), true_sub),
        'BERT Top-1':         top_k_accuracy(sub['sbert_top5'].tolist(), true_sub, k=1),
        'BERT MRR':           mean_reciprocal_rank(sub['sbert_top5'].tolist(), true_sub),
    }

df_lc = pd.DataFrame(lc_results).T
print("=== 생애주기별 성능 ===")
print(df_lc.round(3).to_string())

# 시각화 — Grouped Bar (생애주기 × 모델 × 지표)
fig = go.Figure()
colors = {'TF-IDF': '#4C72B0', 'BERT': '#DD8452'}

for model, col_prefix, color in [('TF-IDF', 'TF-IDF', '#4C72B0'), ('Sentence-BERT', 'BERT', '#DD8452')]:
    fig.add_trace(go.Bar(
        name=f'{model} Top-1',
        x=lifecycle_order,
        y=[lc_results[lc][f'{col_prefix} Top-1'] for lc in lifecycle_order],
        marker_color=color,
        opacity=0.9,
        legendgroup=model,
    ))
    fig.add_trace(go.Bar(
        name=f'{model} MRR',
        x=lifecycle_order,
        y=[lc_results[lc][f'{col_prefix} MRR'] for lc in lifecycle_order],
        marker_color=color,
        opacity=0.5,
        legendgroup=model,
        marker_pattern_shape='/',
    ))

fig.update_layout(
    barmode='group',
    title='생애주기별 TF-IDF vs BERT 성능 비교',
    xaxis_title='생애주기',
    yaxis_title='점수',
    yaxis=dict(range=[0, 1.1]),
    font=dict(size=13),
    legend=dict(orientation='h', y=-0.2),
    width=700, height=420,
)
fig.write_html(str(EVAL_OUTPUT / 'lifecycle_performance.html'))
fig.show()

## 5. 순위 분포 히스토그램 (정답이 몇 위에 나왔는가)

In [ ]:
def get_ranks(pred_list: list[list[int]], true_list: list[int]) -> list[int]:
    """각 쿼리의 정답 순위 반환 (미포함=6으로 처리)."""
    ranks = []
    for pred, true in zip(pred_list, true_list):
        if true in pred:
            ranks.append(pred.index(true) + 1)
        else:
            ranks.append(6)  # Top-5 밖
    return ranks


tfidf_ranks = get_ranks(df['tfidf_top5'].tolist(), true_indices)
sbert_ranks = get_ranks(df['sbert_top5'].tolist(), true_indices)

rank_labels = ['1위', '2위', '3위', '4위', '5위', 'Top-5 밖']
tfidf_dist  = [tfidf_ranks.count(r) for r in range(1, 7)]
sbert_dist  = [sbert_ranks.count(r) for r in range(1, 7)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, dist, label, color in zip(
    axes,
    [tfidf_dist, sbert_dist],
    ['TF-IDF', 'Sentence-BERT'],
    ['#4C72B0', '#DD8452']
):
    bars = ax.bar(rank_labels, dist, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, dist):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title(f'{label} 정답 순위 분포', fontsize=13, fontweight='bold')
    ax.set_xlabel('정답 순위')
    ax.set_ylabel('쿼리 수')
    ax.set_ylim(0, max(max(tfidf_dist), max(sbert_dist)) + 5)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('정답이 몇 순위에 등장하는가 — 순위 분포 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(EVAL_OUTPUT / 'rank_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"TF-IDF   — 1위 정답률: {tfidf_dist[0]}/{len(df)} ({tfidf_dist[0]/len(df)*100:.0f}%)")
print(f"BERT     — 1위 정답률: {sbert_dist[0]}/{len(df)} ({sbert_dist[0]/len(df)*100:.0f}%)")

## 6. TF-IDF 실패 사례 분석 (BERT가 맞추고 TF-IDF가 틀린 케이스)

In [ ]:
# BERT가 Top-1 성공 & TF-IDF가 Top-1 실패한 케이스
bert_wins = []
for i, row in df.iterrows():
    true = row['correct_idx']
    tfidf_ok = true == row['tfidf_top5'][0]
    sbert_ok = true == row['sbert_top5'][0]
    if sbert_ok and not tfidf_ok:
        bert_wins.append({
            'query_id':    row['query_id'],
            'lifeCycle':   row['lifeCycle'],
            'disease':     row.get('disease', ''),
            'query':       row['query'][:80] + '...',
            'tfidf_rank':  tfidf_ranks[i],
            'sbert_rank':  sbert_ranks[i],
        })

df_wins = pd.DataFrame(bert_wins)
print(f"BERT가 TF-IDF를 역전한 케이스: {len(df_wins)}개")
print()
for _, row in df_wins.head(5).iterrows():
    print(f"[{row['query_id']}] {row['lifeCycle']} / {row['disease']}")
    print(f"  쿼리: {row['query']}")
    print(f"  TF-IDF 순위: {row['tfidf_rank']}위 → BERT 순위: {row['sbert_rank']}위")
    print()

## 7. 레이더 차트 — 모델 종합 역량 비교

In [ ]:
radar_metrics = ['Top-1 정확도', 'Top-3 정확도', 'MRR',
                  '자견 Top-1', '성견 Top-1', '노령견 Top-1']

tfidf_radar = [
    metrics['TF-IDF']['Top-1'],
    metrics['TF-IDF']['Top-3'],
    metrics['TF-IDF']['MRR'],
    lc_results['자견']['TF-IDF Top-1'],
    lc_results['성견']['TF-IDF Top-1'],
    lc_results['노령견']['TF-IDF Top-1'],
]
sbert_radar = [
    metrics['Sentence-BERT']['Top-1'],
    metrics['Sentence-BERT']['Top-3'],
    metrics['Sentence-BERT']['MRR'],
    lc_results['자견']['BERT Top-1'],
    lc_results['성견']['BERT Top-1'],
    lc_results['노령견']['BERT Top-1'],
]

# 닫힌 다각형을 위해 첫 값 추가
tfidf_radar += [tfidf_radar[0]]
sbert_radar += [sbert_radar[0]]
radar_metrics_closed = radar_metrics + [radar_metrics[0]]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=tfidf_radar, theta=radar_metrics_closed,
    fill='toself', name='TF-IDF',
    line_color='#4C72B0', fillcolor='rgba(76,114,176,0.2)'
))
fig.add_trace(go.Scatterpolar(
    r=sbert_radar, theta=radar_metrics_closed,
    fill='toself', name='Sentence-BERT',
    line_color='#DD8452', fillcolor='rgba(221,132,82,0.2)'
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='모델 종합 역량 레이더 차트',
    showlegend=True,
    width=550, height=480,
    font=dict(size=13),
)
fig.write_html(str(EVAL_OUTPUT / 'radar_comparison.html'))
fig.show()

## 8. 결과 저장

In [ ]:
EVAL_SUMMARY_PATH = DATA_PROCESSED / 'evaluation_summary.csv'

df_summary = pd.DataFrame({
    '모델':       ['TF-IDF', 'Sentence-BERT'],
    'Top-1':      [metrics['TF-IDF']['Top-1'], metrics['Sentence-BERT']['Top-1']],
    'Top-3':      [metrics['TF-IDF']['Top-3'], metrics['Sentence-BERT']['Top-3']],
    'MRR':        [metrics['TF-IDF']['MRR'],   metrics['Sentence-BERT']['MRR']],
    '자견 Top-1': [lc_results['자견']['TF-IDF Top-1'],   lc_results['자견']['BERT Top-1']],
    '성견 Top-1': [lc_results['성견']['TF-IDF Top-1'],   lc_results['성견']['BERT Top-1']],
    '노령견 Top-1': [lc_results['노령견']['TF-IDF Top-1'], lc_results['노령견']['BERT Top-1']],
})
df_summary.to_csv(EVAL_SUMMARY_PATH, index=False, encoding='utf-8-sig')
print(f"저장 완료: {EVAL_SUMMARY_PATH}")
print()
print(df_summary.round(4).to_string(index=False))

# AWS 마이그레이션 후 S3 업로드
# if is_s3():
#     import boto3
#     s3 = boto3.client('s3', region_name=AWS_REGION)
#     s3.upload_file(str(EVAL_SUMMARY_PATH), S3_BUCKET, 'pet-health-ai/data/processed/evaluation_summary.csv')
#     print("S3 업로드 완료")

## 9. 평가 요약

| 지표 | TF-IDF | Sentence-BERT | 향상 |
|------|--------|--------------|------|
| Top-1 정확도 | (실행 후 기입) | (실행 후 기입) | +?%p |
| Top-3 정확도 | (실행 후 기입) | (실행 후 기입) | +?%p |
| MRR | (실행 후 기입) | (실행 후 기입) | +?%p |

### TF-IDF 한계 및 BERT 우위 근거

**TF-IDF 한계**
- 형태소가 다른 동의어 매칭 불가 (예: "구토" ≠ "토해요" — 정규화 전 쿼리)
- 단어 빈도 기반이라 문맥 이해 없음
- 희소 질병(long-tail)에서 토큰 겹침이 적어 성능 급감

**Sentence-BERT 우위**
- 의미 공간에서 유사한 표현을 가깝게 인코딩
- 구어체 쿼리도 표준 의학 용어와 유사도 높게 매칭
- 생애주기 전반에서 균형 있는 성능 유지

### 생성된 파일
| 파일 | 설명 |
|------|------|
| `data/processed/evaluation_summary.csv` | 지표 요약 테이블 |
| `data/processed/eval_figures/performance_comparison.png` | 전체 성능 막대 그래프 |
| `data/processed/eval_figures/rank_distribution.png` | 순위 분포 히스토그램 |
| `data/processed/eval_figures/lifecycle_performance.html` | 생애주기별 성능 인터랙티브 |
| `data/processed/eval_figures/radar_comparison.html` | 레이더 차트 |

## 10. SBERT 임베딩 t-SNE 시각화 (연구질문 2 시각적 증거)

In [ ]:
from sklearn.manifold import TSNE
import plotly.express as px

# 시각화용 샘플 (전체 임베딩이 크면 느림 → 최대 1,000개 샘플링)
SAMPLE_N = min(1000, len(df_db))
sample_idx = np.random.default_rng(42).choice(len(df_db), SAMPLE_N, replace=False)

sample_emb  = db_embeddings[sample_idx]
sample_meta = df_db.iloc[sample_idx].reset_index(drop=True)

# t-SNE 2D 축소
print("t-SNE 계산 중 (1~2분)...")
tsne   = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=1000)
coords = tsne.fit_transform(sample_emb)

df_tsne = pd.DataFrame({
    'x':          coords[:, 0],
    'y':          coords[:, 1],
    'lifeCycle':  sample_meta['lifeCycle'],
    'department': sample_meta['department'],
    'disease':    sample_meta['disease'],
    'text':       sample_meta['input'].str[:60] + '...',
})

# 생애주기별 산점도
fig = px.scatter(
    df_tsne, x='x', y='y',
    color='lifeCycle',
    hover_data=['department', 'disease', 'text'],
    color_discrete_map={'자견': '#4C72B0', '성견': '#55A868', '노령견': '#DD8452'},
    title='SBERT 임베딩 t-SNE 시각화 — 생애주기별 색상<br>'
          '<sup>개념 모델의 LifeCycle 클래스 라벨 기준. 클러스터 형성 여부가 의미 매칭의 시각적 근거.</sup>',
    opacity=0.7,
    width=750, height=550,
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(font=dict(size=12))
fig.write_html(str(EVAL_OUTPUT / 'tsne_lifecycle.html'))
fig.show()

# 진료과별 산점도
fig2 = px.scatter(
    df_tsne, x='x', y='y',
    color='department',
    hover_data=['lifeCycle', 'disease', 'text'],
    title='SBERT 임베딩 t-SNE 시각화 — 진료과별 색상<br>'
          '<sup>개념 모델의 Department 클래스 라벨 기준.</sup>',
    opacity=0.7,
    width=750, height=550,
)
fig2.update_traces(marker=dict(size=5))
fig2.write_html(str(EVAL_OUTPUT / 'tsne_department.html'))
fig2.show()

print("t-SNE 시각화 저장 완료")